# Your scraper has an accent

Two clients. Same machine, same address, same URL. One gets a block page and
the other gets the site.

The only difference between them is what their TLS handshake and HTTP/2
connection look like on the wire. No proxy, no JavaScript, no captcha solver.

In [ ]:
import html
import json
import re

import httpx
from IPython.display import HTML, display

from pybrws.client import get

PROBE = "https://tls.peet.ws/api/all"
BLOCKED = (403, 503)

# Bright enough for a projector, in either notebook theme.
GOOD, BAD, WARN, DIM = "#2ea043", "#f85149", "#d29922", "#8b949e"
CELL = "padding:7px 18px 7px 0;white-space:nowrap;text-align:left;"


def fetch(client, url):
    """Return (status, body) from either client, or (None, error text)."""
    try:
        if client == "pybrws":
            response = get(url, follow_redirects=5, timeout=30)
            return response.status, response.text()
        with httpx.Client(http2=True, follow_redirects=True, timeout=30) as http:
            response = http.get(url)
        return response.status_code, response.text
    except Exception as error:
        return None, "%s: %s" % (type(error).__name__, error)


def page_title(page):
    found = re.search(r"<title[^>]*>(.*?)</title>", page, re.S | re.I)
    return html.escape(found.group(1).strip()[:70]) if found else "(no title)"


def show(label, status, page, height=360):
    """Summarise a result, then preview it in a sandboxed frame.

    The summary is plain markup so it survives a renderer that strips
    iframes; the preview is a bonus where frames are allowed.
    """
    served = status is not None and status not in BLOCKED and status != 429
    tone = GOOD if served else BAD
    display(HTML(
        '<div style="font:14px/1.6 system-ui;padding:10px 14px;border-radius:6px;'
        'border:1px solid %s"><b>%s</b>'
        '<span style="color:%s;font-weight:700;margin-left:12px">%s</span>'
        '<span style="color:%s;margin-left:12px">%s bytes</span>'
        '<div style="color:%s;margin-top:4px">&lt;title&gt; %s</div></div>'
        % (tone, html.escape(label), tone,
           "HTTP %s" % status if status else "failed",
           DIM, format(len(page), ","), DIM, page_title(page))
    ))
    display(HTML(
        '<iframe sandbox="" loading="lazy" style="width:100%%;height:%dpx;'
        'border:1px solid %s;border-radius:6px;background:#fff;margin-top:6px" '
        'srcdoc="%s"></iframe>' % (height, DIM, html.escape(page, quote=True))
    ))


def compare(url, height=360):
    for client in ("httpx", "pybrws"):
        status, page = fetch(client, url)
        show("%s  %s" % (client, url), status, page, height)


def table(header, rows):
    head = "".join('<th style="%s">%s</th>' % (CELL, h) for h in header)
    body = "".join("<tr>%s</tr>" % "".join(r) for r in rows)
    display(HTML(
        '<div style="overflow-x:auto"><table style="font:13px/1.4 ui-monospace,'
        'SFMono-Regular,Menlo,monospace;border-collapse:collapse">'
        '<tr style="border-bottom:1px solid %s;color:%s">%s</tr>%s</table></div>'
        % (DIM, DIM, head, body)
    ))


def td(text, colour=None, bold=False):
    return '<td style="%scolor:%s;%s">%s</td>' % (
        CELL, colour or "inherit", "font-weight:700" if bold else "", text)

## 1. zillow.com

`httpx` is a mature HTTP/2 client over Python's standard TLS stack. `pybrws`
differs from it only in the bytes of its ClientHello, its HTTP/2 preamble and
its header order.

In [ ]:
compare("https://www.zillow.com/")

## 2. etsy.com, behind DataDome

A category page rather than the homepage. The strictest rules sit on the
endpoints most worth scraping: etsy refuses `/` and `/?q=` to this client too,
while serving `/c/jewelry`.

In [ ]:
compare("https://www.etsy.com/c/jewelry")

## 3. A wider sweep

Including the targets where this does not work, and a control that serves
everyone. A row where both clients fail is not a failure of the measurement.

`429` is rate limiting earned by a day of measuring, not a verdict on the
client, so it is marked apart from a block.

In [ ]:
TARGETS = [
    ("https://www.zillow.com/", ""),
    ("https://www.etsy.com/c/jewelry", ""),
    ("https://www.glassdoor.com/", "not deterministic, flaps between runs"),
    ("https://www.etsy.com/", "homepage, stricter rule than the category page"),
    ("https://www.g2.com/", ""),
    ("https://example.com/", "control, serves everyone"),
]


def verdict(status):
    if status is None:
        return BAD, "&#10007; failed"
    if status == 429:
        return WARN, "&#8211; 429 rate limited"
    if status in BLOCKED:
        return BAD, "&#10007; %d" % status
    return GOOD, "&#10003; %d" % status


rows = []
for url, note in TARGETS:
    cells = [td(html.escape(url))]
    for client in ("pybrws", "httpx"):
        colour, text = verdict(fetch(client, url)[0])
        cells.append(td(text, colour, bold=True))
    cells.append(td(note, DIM))
    rows.append(cells)

table(["target", "pybrws", "httpx", ""], rows)

## 4. What the server actually saw

`tls.peet.ws` reports back the fingerprint of whoever asked. The first row is
a real Firefox 157, captured by opening that URL in the browser.

In [ ]:
with open("fixtures/reference-firefox-157-peet.json", encoding="utf-8") as handle:
    reference = json.load(handle)

with httpx.Client(http2=True, timeout=30) as http:
    seen_baseline = http.get(PROBE).json()
seen_ours = json.loads(get(PROBE, timeout=30).body)

target_ja4 = reference["tls"]["ja4"]
target_h2 = reference["http2"]["akamai_fingerprint"]

rows = []
for index, (name, seen) in enumerate(
    [("Firefox 157", reference), ("pybrws", seen_ours), ("httpx", seen_baseline)]
):
    cells = [td(name, bold=True)]
    for value, target in ((seen["tls"]["ja4"], target_ja4),
                          (seen["http2"]["akamai_fingerprint"], target_h2)):
        if index == 0:
            cells.append(td(html.escape(value), DIM))
        else:
            match = value == target
            mark = "&#10003; " if match else "&#10007; "
            cells.append(td(mark + html.escape(value), GOOD if match else BAD))
    rows.append(cells)

table(["client", "JA4", "HTTP/2 fingerprint"], rows)

## 5. The part that is easy to oversell

The fingerprint is necessary and it is not sufficient. Crossing it against the
address it comes from, three rounds each:

| | pybrws | httpx |
|---|---|---|
| **home connection** | 3/3 served | 0/3 |
| **commercial ISP proxy** | 0/3 | 0/3 |

One cell of four passes. A proxy address sold as clean is itself a signal, and
here it scored worse than an ordinary domestic line.

And nothing here solves a JavaScript challenge. A site that issues one is out
of reach for any HTTP client, by construction.